# 01 - Exploracion del dataset MangoDHDS

Conteo por clase, muestreo visual, verificacion de que todas las labels son bboxes de imagen completa, y chequeo de fuga entre splits.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.config import CLASSES, DATASET_MANGODHDS_ROOT
print('Repo root:', ROOT)
print('Dataset:', DATASET_MANGODHDS_ROOT)

In [ ]:
from collections import Counter
import pandas as pd

SPLITS = ['train', 'valid', 'test']
rows = []
for split in SPLITS:
    labels_dir = DATASET_MANGODHDS_ROOT / split / 'labels'
    counter = Counter()
    for txt in labels_dir.glob('*.txt'):
        for line in txt.read_text().splitlines():
            s = line.strip()
            if s:
                counter[int(s.split()[0])] += 1
    for cls_id, n in sorted(counter.items()):
        rows.append({'split': split, 'class_id': cls_id, 'class': CLASSES[cls_id], 'n': n})
pd.DataFrame(rows)

In [ ]:
# Verificacion del hallazgo critico
full_image = 0
other = 0
for split in SPLITS:
    for txt in (DATASET_MANGODHDS_ROOT / split / 'labels').glob('*.txt'):
        for line in txt.read_text().splitlines():
            s = line.strip()
            if not s:
                continue
            parts = s.split()
            if len(parts) == 5 and parts[1:] == ['0.5', '0.5', '1.0', '1.0']:
                full_image += 1
            else:
                other += 1
pct = 100 * full_image / max(1, full_image + other)
print(f'HALLAZGO: el {pct:.1f}% de las labels son bboxes de imagen completa.')
print(f'Este dataset es efectivamente clasificacion, no deteccion.')
print(f'full_image={full_image}  other={other}')

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 3 * len(CLASSES)))
for row, cls in enumerate(CLASSES):
    samples = []
    for split in SPLITS:
        labels_dir = DATASET_MANGODHDS_ROOT / split / 'labels'
        for txt in labels_dir.glob('*.txt'):
            line = txt.read_text().splitlines()[0].strip()
            if int(line.split()[0]) == row:
                img_path = DATASET_MANGODHDS_ROOT / split / 'images' / (txt.stem + '.jpg')
                if img_path.is_file():
                    samples.append(img_path)
    random.seed(42)
    random.shuffle(samples)
    for col, path in enumerate(samples[:4]):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls if col == 0 else '')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Chequeo de fuga de datos entre splits
!python src/data_prep/check_leakage.py